# 0. Import all core libraries

In [1]:
# Core Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import timm  # PyTorch Image Models - has Swin Transformer

# Optimizers and Schedulers
from torch.optim import Adam, AdamW, SGD
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

# Data Handling
import numpy as np
import pandas as pd
from PIL import Image
import cv2

# Dataset and Preprocessing
from sklearn.model_selection import train_test_split

# Class Imbalance (SMOTE)
from imblearn.over_sampling import SMOTE
from collections import Counter

# Metrics and Evaluation
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    f1_score,
    recall_score
)

# Grad-CAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

# For reproducibility
import random
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

C:\Users\aria_\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


# 1. Define my model's components

## 1.1 Turning the Swin patch embedding layer into a PyTorch module
1. Create a class called "PatchEmbedding" that inherits from "nn.Module"
2. Initialise with appropriate hyperparameters, such as channels, embedding dimension, patch size.
3. Create a layer to turn an image into embedded patches using "nn.Conv2d()"
4. Create a layer to flatten the feature maps of the output of the layer
5. Define a "forward()" that defines the forward computation (e.g. pass through layer from 3 and 4)
6. Make sure the output shape of the layer reflects the required output shape of the patch embedding

In [2]:
# Step 1
class PatchEmbedding(nn.Module):
    # Step 2
    def __init__(self,
                 in_channels:int=3,
                 embedding_dim:int=128, # Swin-Base's embedding dimensions
                 patch_size:int=4):
        super(PatchEmbedding, self).__init__()

        self.patch_size = patch_size
        self.embedding_dim = embedding_dim

        # Step 3
        self.patcher = nn.Conv2d(in_channels=in_channels,
                                out_channels=embedding_dim,
                                kernel_size=patch_size,
                                stride=patch_size,
                                padding=0)

        # Step 4
        self.flatten = nn.Flatten(start_dim=2,
                                  end_dim=3)

        self.norm = nn.LayerNorm(embedding_dim)

    #Step 5
    def forward(self, x):
        # Create assertion to check that inputs are the correct shape
        image_resolution = x.shape[-1]
        assert image_resolution % self.patch_size == 0, \
            f"Input image size must be divisible by patch size, \
            image shape: {image_resolution}, patch size: {self.patch_size}"

        # Perform forward pass
        x_patched = self.patcher(x)
        x_flattened = self.flatten(x_patched)

        # Step 6
        x = x_flattened.permute(0, 2, 1)
        x = self.norm(x)

        return x

## 1.2 Custom Depthwise Separable Convolution and Feedforward Blocks
The Depthwise Separable FeedForward Network, in accordance with my proposal:
1. Changes the dimensions of the sequential transformer output into a spatial grid of patches that nn.Conv2d can accept for depthwise convolutions
2. Perforsm two 3*3 depthwise convolutions, followed by a Gaussian Error Linear Uni (GELU) and a pointwise convolution, which expands and diminishes the channel quantity by a factor of 4 after the first and second blocks respectively, and dropout by a rate of 0.1 (subject to change).
3. The spatial grid can then be reshaped and transposed back into its original sequence that can then be further used by the Swin Transformer

In [3]:
class DepthwiseSeparableConv(nn.Module):
    """"Depthwise Separable Convolution block for Swin-Xception FFN"""
    def __init__(self,
                 in_channels,
                 out_channels,
                 kernel_size:int=3):
        super(DepthwiseSeparableConv, self).__init__()
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels, # The quantity of output channels must remain the same as input channels for depthwise convolutions
            kernel_size,
            padding=kernel_size//2,
            groups=in_channels # Gives each input channel a separate spatial filter
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1) # Combines each separated, filtered channel
        self.gelu = nn.GELU() # Gaussian Error Linear Unit inbetween depthwise and pointwise convolutions

    def forward(self, x): # A step forward entails a 3 * 3 depthwise convolution, followed by GELU and a pointwise convolution
        x = self.depthwise(x)
        x = self.gelu(x)
        x = self.pointwise(x)

        return x

class DepthwiseSeparableFFN(nn.Module):
    """Replaces the standard MLP for Swin Transformers with depthwise separable convolutions"""
    def __init__(self,
                embedding_dim,
                mlp_ratio:int=4, #Expands the quantity of channels by a factor of 4
                dropout:float=0.1):
        super(DepthwiseSeparableFFN, self).__init__()

        hidden_dim = int(embedding_dim * mlp_ratio) #the hidden dimensions are number of embeddings for Swin-Base, multiplied by a factor of 4

        self.ds_conv1 = DepthwiseSeparableConv(embedding_dim, hidden_dim, kernel_size=3)
        self.dropout1 = nn.Dropout(dropout)

        self.ds_conv2 = DepthwiseSeparableConv(hidden_dim, embedding_dim, kernel_size=3)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        # Step 1
        # Swin Transformers take a sequence of inputs,
        B, N, C = x.shape

        H = W = int(np.sqrt(N))

        x = x.transpose(1, 2).reshape(B, C, H, W)

        # Step 2
        # Perform depthwise separable convolutions, with appropriate dropout
        x = self.ds_conv1(x) #Becomes [B, 4C, H, W]
        x = self.dropout1(x)
        x = self.ds_conv2(x) #Becomes [B, C, H, W]
        x = self.dropout2(x)

        x = x.reshape(B, C, N).transpose(1, 2)
        
        return x

## 1.3 Utilising my DSFFN to create a Swin-Xception block
My proposed Swin-Xception block follows this procedure:
1. Layer Normalisation
2. Windowed Multi-Head Self Attention
3. Add the original input to the output of the attention
4. Layer Normalisation
5. Depthwise Separable FeedForward Network
6. Add Step 3 to the output from the DS-FFN

In [4]:
class SwinXceptionBlock(nn.Module):
    """Swin Transformer block that uses depthwise separable FFN instead of an MLP"""
    def __init__(self,
                 embedding_dim,
                 num_heads,
                 input_resolution,
                 window_size:int=7,
                 shift_size=0,
                 mlp_ratio=4):
        super(SwinXceptionBlock, self).__init__()

        from timm.models.swin_transformer import SwinTransformerBlock

        self.block = SwinTransformerBlock( # Use timm's Swin Transformer Block...
            dim=embedding_dim,
            input_resolution=input_resolution,
            num_heads=num_heads,
            window_size=window_size,
            shift_size=shift_size,
            mlp_ratio=mlp_ratio
        )

        self.input_resolution = input_resolution

        self.block.mlp = DepthwiseSeparableFFN(embedding_dim, mlp_ratio) # But replace the mlp head with my DS-FFN

    def forward(self, x): # X input is sequential transformer data
        B, N, C = x.shape

        H, W = self.input_resolution # height and width

        x = x.view(B, H, W, C)

        # Swin blocks accept inputs as spatial grid
        x = self.block(x)

        # back to sequential data
        x = x.view(B, N, C)
        
        return x

## 1.4 Creating a Patch Merging block
My Patch Merging block reduces token length by a factor of 4, and increases the number of channels by a factor of 2:
1. Take Sequential Token input from Swin
2. Rearrange the the input to achieve a spatial grid of token "patches"
3. Split the grid into a non-overlapping 2*2 selection of tokens
4. Concatenate the spatial grid (increases channels by a factor of 4)
5. Linear project halfs the spatial grid and doubles channels

In [5]:
class PatchMerging(nn.Module):
    """"Merges 2 * 2 neighbouring patches, reducing spatial dimensions by a factor 2"""
    def __init__(self, embedding_dim):
        super(PatchMerging, self).__init__()
        self.embedding_dim = embedding_dim
        self.reduction = nn.Linear(4*embedding_dim, 2*embedding_dim, bias=False) # Reduces channels
        self.norm = nn.LayerNorm(4*embedding_dim)
        

    def forward(self, x):
        """
        x: [B, N, C] Batch Number (B), Patch Count (N = H * W), Channels (C)
        Returns : [B, (H/2)*(W/2), 2*C]
        My patch merging block quarters the size and doubles the number of channels
        """
        B, N, C = x.shape # Step 1
        H = W = int(np.sqrt(N))
        
        assert H % 2 == 0 and W % 2 == 0, f"H and W must be even, got H={H}, W={W}"

        x = x.view(B, H, W, C) # Step 2

        # Step 3, split into four groups
        x0 = x[:, 0::2, 0::2, :] # top left
        x1 = x[:, 1::2, 0::2, :] # bottom left
        x2 = x[:, 0::2, 1::2, :] # top right
        x3 = x[:, 1::2, 1::2, :] # bottom right

        # Step 4, Concatenate along channel dimension
        x = torch.cat([x0, x1, x2, x3], dim=-1)

        # Flatten back to sequence for Swin
        x = x.view(B, -1, 4*C)

        # Step 5
        x = self.norm(x)
        x = self.reduction(x)
        
        return x

## 1.5 Define my Swin-Xception backbone, combining all previous modules
The Swin-Xception backbone follows this structure:
1. Patch Embedding
2. Stage 1: 2 Swin-Xception blocks
3. Patch Merging
4. Stage 2: 2 Swin-Xception blocks
5. Patch Merging
6. Stage 3: 6 Swin-Xception blocks
7. Patch Merging
8. Stage 4: 2 Swin-Xception blocks
9. Global Average Pooling
10. Output Features

This module also covers the MLP Head that follows the Global Average Pooling

In [6]:
class SwinXception(nn.Module):
    """Swin backbone that uses depthwise separable FFN instead of the standard MLP"""
    def __init__(self, num_classes=7):
        super(SwinXception, self).__init__()

        self.patch_embed = PatchEmbedding(in_channels=3, embedding_dim=128, patch_size=4)

        # Stage 1
        # nn.ModuleList runs consecutive modules in order. In this case, the list comprehension runs
        # two SwinXception blocks, as specified in my proposal
        self.stage1 = nn.ModuleList([SwinXceptionBlock(128, num_heads=4, input_resolution=(56, 56), shift_size=0 if i % 2 == 0 else 3) for i in range(2)])
        self.merge1 = PatchMerging(128)

        # Stage 2
        self.stage2 = nn.ModuleList([SwinXceptionBlock(256, num_heads=8, input_resolution=(28, 28), shift_size=0 if i % 2 == 0 else 3) for i in range (2)])
        self.merge2 = PatchMerging(256)

        # Stage 3
        # in range (6) details the six SwinXception blocks at stage 3
        self.stage3 = nn.ModuleList([SwinXceptionBlock(512, num_heads=16, input_resolution=(14, 14), shift_size=0 if i % 2 == 0 else 3) for i in range (6)])
        self.merge3 = PatchMerging(512)
        
        # Stage 4
        self.stage4 = nn.ModuleList([SwinXceptionBlock(1024, num_heads=32, input_resolution=(7, 7), shift_size=0 if i % 2 == 0 else 3) for i in range (2)])

        # Average Pooling
        self.norm = nn.LayerNorm(1024)
        self.avgpool1d = nn.AdaptiveAvgPool1d(output_size=1)

        # MLP head
        self.head = nn.Linear(1024, num_classes)
        
    def forward(self, x):
        # x = [B, 3, 224, 224]
        x = self.patch_embed(x) # x = [B, 3136, 128]

        # Stage 1: 2 Swin Blocks
        for block in self.stage1:
            x = block(x) 
        x = self.merge1(x) # x = [B, 784, 256]

        # Stage 2: 2 Swin Blocks
        for block in self.stage2:
            x = block(x)
        x = self.merge2(x) # x = [B, 196, 512]

        # Stage 3: 6 Swin Blocks
        for block in self.stage3:
            x = block(x)
        x = self.merge3(x) # x = [B, 49, 1024]

        # Stage 4: 2 Swin Blocks
        for block in self.stage4:
            x = block(x)

        # Normalisation and Average Pooling
        x = self.norm(x)
        x = x.transpose(1, 2) # x = [B, 1024, 49]
        x = self.avgpool1d(x) # x = [B, 1024, 1]
        x = torch.flatten(x, 1) # x = [B, 1024]

        #MLP Head
        x = self.head(x) #x = [B, 7]

        return x

# 2. Data Preprocessing

## 2.1 Data Preprocessing function and loading datasets
Data preprocessing in pytorch can be handled as follows:
torch.utils.data has Dataset and DataLoader objects, the former finds the directory of the dataset, while the latter accommodates the loading of data in batches, as well as facilitates random loading order

In [7]:
import sys

# Get the root of the notebook directory
NOTEBOOK_DIR = os.getcwd()

# Construct the path to src
SRC_PATH = os.path.abspath("../../src")

# Add it to sys.path
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

# Now import
from datasets import FERDataset

# torchvision augmentations to change orientation and color; each image is of fixed size and depth (224 * 224 * 3)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# torchvision augmentations for test images, deciding not to change as many features
transform_test = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = FERDataset('../../datasets/FER2013/train', transform=transform_train)
test_fer = FERDataset('../../datasets/FER2013/test', transform=transform_test)
test_raf = FERDataset('../../datasets/RAFDB/DATASET/test', transform=transform_test)

# torch.utils.data DataLoaders to aid in batch processing, and the order of images to be trained/tested on
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
test_fer_loader = DataLoader(test_fer, batch_size=32, shuffle=False, num_workers=4)
test_raf_loader = DataLoader(test_raf, batch_size=32, shuffle=False, num_workers=4)

print(f"Train samples: {len(train_dataset)}")
print(f"FER2013 test samples: {len(test_fer)}")
print(f"RAF-DB test samples: {len(test_raf)}")

Train samples: 28709
FER2013 test samples: 7178
RAF-DB test samples: 3068


# 3. Stage 1: End-to-End Training

## 3.1 Define a training function

In [8]:
def train_one_epoch(model, dataloader, criterion, optimiser, device):
    """" Used for the train dataset, runs the train function and calculates loss and accuracy """
    
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        # Move images and labels to device
        images = images.to(device)
        labels = labels.to(device)

        # Zero gradients
        optimiser.zero_grad()

        # Forward pass
        outputs = model(images) # [B, 7]

        loss = criterion(outputs, labels) # Cross-Entropy Loss

        # Backpropagation
        loss.backward()
        optimiser.step()

        # metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

## 3.2 Validate function

In [9]:
def validate(model, dataloader, criterion, device):
    """ Used for the test splits in FER2013 and RAF-DB, runs the model evaluation and calculates loss and accuracy for that epoch """
    
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(dataloader) # loss score, output each epoch
    epoch_acc = 100. * correct / total # accuracy is also reported each epoch
    return epoch_loss, epoch_acc

## 3.3 Model Training

In [10]:
PATH = "checkpoints/latest.pth"
os.makedirs("checkpoints", exist_ok=True)

model = SwinXception(num_classes=7).to(device) # signifies the device being ran on (cuda)
criterion = nn.CrossEntropyLoss()

# AdamW for decoupled weight decay and increased countermeasures towards overfitting
optimiser = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Cosine Annealing for fast initial learning and a gentler convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=50)

start_epoch = 0

if os.path.exists(PATH): # resumes most recent epoch if training isinterrupted midway
    print(f"Loading checkpoint from {PATH}")
    checkpoint = torch.load(PATH, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimiser.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
else:
    print("No checkpoint found — starting from scratch")

num_epochs = 50
for epoch in range(start_epoch, num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimiser, device)

    fer_loss, fer_acc = validate(model, test_fer_loader, criterion, device)
    raf_loss, raf_acc = validate(model, test_raf_loader, criterion, device)

    scheduler.step()

    print(f"Training Loss: {train_loss:.4f}, and accuracy: {train_acc:.2f}%")
    print(f"FER2013 Validation Loss: {fer_loss:.4f}, and accuracy: {fer_acc:.2f}%")
    print(f"RAF-DB Validation Loss: {raf_loss:.4f}, and accuracy: {raf_acc:.2f}%")

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimiser.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
    }

    torch.save(checkpoint, PATH)

No checkpoint found — starting from scratch
Epoch 1/50


 10%|████████                                                                         | 90/898 [00:41<06:13,  2.16it/s]


KeyboardInterrupt: 

# 4. Stage 2: Feature Extraction and Synthesis

## 4.1 Freeze Swin-Xception backbone, but keep the MLP head modifiable

In [17]:
# setting 'requires_grad' on each parameter of a block to False signifies that said parameter
# won't change from the gradient, *freezing* its learning

# Freeze the backbone
for param in model.patch_embed.parameters():
    param.requires_grad = False
for stage in [model.stage1, model.stage2, model.stage3, model.stage4]:
    for block in stage:
        for param in block.parameters():
            param.requires_grad = False

# Keep MLP head trainable
for param in model.head.parameters():
    param.requires_grad = True

# To freeze a block, freeze all of its parameters
# To freeze a parameter, set requires_grad = False

## 4.2 Create a function to extract deep features processed by the frozen model

In [21]:
def extract_features(model, dataloader, device):
    model.eval()
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Extracting features"):
            images = images.to(device)

            x = model.patch_embed(images)

            for block in model.stage1:
                x = block(x)
            x = model.merge1(x)

            for block in model.stage2:
                x = block(x)
            x = model.merge2(x)

            for block in model.stage3:
                x = block(x)
            x = model.merge3(x)

            for block in model.stage4:
                x = block(x)

            x = model.norm(x)
            x = x.transpose(1, 2)
            x = model.avgpool1d(x)
            x = torch.flatten(x, 1)

            all_features.append(x.cpu().numpy())
            all_labels.append(labels.numpy())

        features = np.vstack(all_features)
        labels = np.concatenate(all_labels)

        return features, labels

## 4.3 Function to apply SMOTE to minority classes

In [22]:
def apply_smote(features, labels, random_state=42):
    print("Original class distribution:")
    print(Counter(labels))

    smote = SMOTE(random_state=random_state, k_neighbors=5)
    features_balanced, labels_balanced = smote.fit_resample(features, labels)

    print("\nBalanced class distribution:")
    print(Counter(labels_balanced))

    return features_balanced, labels_balanced

In [23]:
features, labels = extract_features(model, train_loader, device)
features_balanced, labels_balanced = apply_smote(features, labels)

Extracting features: 100%|███████████████████████████████████████████████████████████| 898/898 [02:27<00:00,  6.08it/s]


Original class distribution:
Counter({np.int64(3): 7215, np.int64(4): 4965, np.int64(5): 4830, np.int64(2): 4097, np.int64(0): 3995, np.int64(6): 3171, np.int64(1): 436})

Balanced class distribution:
Counter({np.int64(2): 7215, np.int64(3): 7215, np.int64(5): 7215, np.int64(6): 7215, np.int64(0): 7215, np.int64(4): 7215, np.int64(1): 7215})


# 5. Stage 3: Classifier Retraining on balanced set

## 5.1 Function to retrain only the MLP head

In [27]:
def retrain_mlp_head(model, features, labels, device, epochs=10, batch_size=256, lr=1e-3):
    # setting 'requires_grad' on each parameter of a block to False signifies that said parameter
    # won't change from the gradient, *freezing* its learning

    # Freeze the backbone
    for param in model.patch_embed.parameters():
        param.requires_grad = False
    for stage in [model.stage1, model.stage2, model.stage3, model.stage4]:
        for block in stage:
            for param in block.parameters():
                param.requires_grad = False

    # Keep MLP head trainable
    for param in model.head.parameters():
        param.requires_grad = True

    # To freeze a block, freeze all of its parameters
    # To freeze a parameter, set requires_grad = False

    # Create dataset from features
    features_tensor = torch.FloatTensor(features)
    labels_tensor = torch.LongTensor(labels)
    dataset = torch.utils.data.TensorDataset(features_tensor, labels_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle = True)

    criterion = nn.CrossEntropyLoss()
    optimiser = torch.optim.Adam(model.head.parameters(), lr=lr)

    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_features, batch_labels in tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}"):
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            optimiser.zero_grad()

            outputs = model.head(batch_features)
            loss = criterion(outputs, batch_labels)

            loss.backward()
            optimiser.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += batch_labels.size(0)
            correct += predicted.eq(batch_labels).sum().item()

        epoch_loss = running_loss / len(dataloader)
        epoch_acc = 100. * correct / total

        print(f"Epoch {epoch+1}: Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

    return model

## 5.2 Train MLP Head

In [28]:
model = retrain_mlp_head(
    model,
    features_balanced,
    labels_balanced,
    device,
    epochs=10,
    batch_size=256,
    lr=1e-3
)

torch.save(model.state_dict(), 'swin_xception_final.pth')

Epoch 1/10: 100%|████████████████████████████████████████████████████████████████████| 198/198 [00:02<00:00, 97.63it/s]


Epoch 1: Loss: 0.0055, Accuracy: 99.85%


Epoch 2/10: 100%|███████████████████████████████████████████████████████████████████| 198/198 [00:01<00:00, 166.35it/s]


Epoch 2: Loss: 0.0051, Accuracy: 99.84%


Epoch 3/10: 100%|███████████████████████████████████████████████████████████████████| 198/198 [00:01<00:00, 152.81it/s]


...

Epoch 8: Loss: 0.0040, Accuracy: 99.87%


Epoch 9/10:  90%|████████████████████████████████████████████████████████████▏      | 178/198 [00:01<00:00, 121.85it/s]Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x000001E371E4FC40>
Traceback (most recent call last):
  File "C:\Users\aria_\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "C:\Users\aria_\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py", line 1612, in _shutdown_workers
    if self._persistent_workers or self._workers_status[worker_id]:
                                   ^^^^^^^^^^^^^^^^^^^^
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'
Epoch 9/10: 100%|████████████████████████████████████████████████████████████████████| 198/198 [00:02<00:00, 83.03it/s]


Epoch 9: Loss: 0.0038, Accuracy: 99.88%


Epoch 10/10: 100%|██████████████████████████████████████████████████████████████████| 198/198 [00:01<00:00, 142.82it/s]


Epoch 10: Loss: 0.0041, Accuracy: 99.86%


# 6. Final Evaluation

In [29]:
fer_loss, fer_acc = validate(model, test_fer_loader, criterion, device)
raf_loss, raf_acc = validate(model, test_raf_loader, criterion, device)

print(f"FER2013 Test Accuracy: {fer_acc:.2f}%")
print(f"RAF-DB Test Accuracy: {raf_acc:.2f}%")

FER2013 Test Accuracy: 58.78%
RAF-DB Test Accuracy: 34.22%
